# Show databricks resources
**Purpose:** Debug utility — inspect secret scopes, secrets, mounts, and cluster config.

In [ ]:
# List all secret scopes with full detail (name, backend type, Key Vault resource_id + dns_name)
import requests, json

ctx   = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
token = ctx.apiToken().get()
host  = ctx.apiUrl().get()

response = requests.get(
    f"{host}/api/2.0/secrets/scopes/list",
    headers={"Authorization": f"Bearer {token}"}
)

data = response.json()
print(json.dumps(data, indent=2))

# Also show via dbutils for quick name check
# Note: dbutils SecretScope only exposes .name — backend_type is REST API only
print("\n── dbutils view ──")
for s in dbutils.secrets.listScopes():
    print(f"  scope={s.name}")

In [ ]:
# List secrets inside kv-scope (names only — values are redacted by Databricks)
display(dbutils.secrets.list("kv-scope"))

In [ ]:
# Verify individual secrets are readable (shows length, not value)
for key in ["sp-client-id", "sp-client-secret", "sp-tenant-id", "sql-username", "sql-password", "databricks-token"]:
    try:
        val = dbutils.secrets.get(scope="kv-scope", key=key)
        print(f"✅ {key}: {len(val)} chars")
    except Exception as e:
        print(f"❌ {key}: {e}")

In [ ]:
# Show Spark config — verify OAuth2 entries after storagemount runs
for k, v in spark.sparkContext.getConf().getAll():
    if "azure" in k.lower():
        print(f"{k} = {'*****' if 'secret' in k.lower() else v}")